# Karachay-Balkar (krc) — Tokenisation and Morphological Analysis

Karachay-Balkar (Kipchak branch) is supported via Cyrillic script tokenisation and Prototype-quality Apertium FST morphological analysis. NLLB-200 provides cross-lingual embeddings and machine translation.

In [ ]:
# Install TurkicNLP
# pip install turkicnlp          # core (tokenization, transliteration)
# pip install "turkicnlp[stanza]"  # adds POS, lemma, depparse, NER
# pip install "turkicnlp[nllb]"    # adds cross-lingual embeddings + translation
# pip install "turkicnlp[all]"     # all optional dependencies

In [ ]:
import turkicnlp
from turkicnlp import Pipeline

## 1. Download Models

In [ ]:
turkicnlp.download('krc')

## 2. Tokenisation

In [ ]:
from turkicnlp.scripts import Script
from turkicnlp.scripts.detector import detect_script
from turkicnlp.scripts.transliterator import Transliterator

# Karachay-Balkar Cyrillic text
cyrl = "Мен школгъа барама."
print("Script Detection:")
print(f"  Detected: {detect_script(cyrl).name}")
print()

# Cyrillic -> Turkic Common Alphabet (Latin)
try:
    t = Transliterator("krc", source=Script.CYRILLIC, target=Script.COMMON_TURKIC)
    common = t.transliterate(cyrl)
    print(f"Cyrillic:        {cyrl}")
    print(f"Turkic Common:   {common}")
    
    # Reverse: Turkic Common -> Cyrillic
    t_back = Transliterator("krc", source=Script.COMMON_TURKIC, target=Script.CYRILLIC)
    cyrl_restored = t_back.transliterate(common)
    print(f"Restored:        {cyrl_restored}")
    print(f"Round-trip match: {cyrl == cyrl_restored}")
except Exception as e:
    print(f"⚠ Note: Transliteration to Turkic Common Alphabet (Latin) may not be fully supported for Karachay-Balkar: {e}")
    print("  For Cyrillic-based languages, use Script.LATIN as alternative")

## 2. Script Detection and Cyrillic ↔ Latin Transliteration

Karachay-Balkar uses Cyrillic script. The transliteration system enables conversion to Latin for cross-script analysis.

In [ ]:
nlp_tok = Pipeline("krc", processors=["tokenize"])
doc = nlp_tok("Мен школгъа барама.")
print([w.text for w in doc.words])

## 3. Morphological Analysis (Apertium FST — Prototype)

In [ ]:
nlp = Pipeline(
    "krc",
    processors=["tokenize", "morph"],
    morph_backend="apertium",
)
doc = nlp("Мен школгъа барама.")
for w in doc.words:
    print(f"{w.text:<18} lemma={w.lemma:<12} feats={w.feats}")

## 4. Translation via NLLB-200

In [ ]:
turkicnlp.download("krc", processors=["translate"])
trans = Pipeline("krc", processors=["translate"], translate_tgt_lang="eng_Latn")
doc = trans("Мен школгъа барама.")
print("EN:", doc.translation)